In [1]:
import pandas as pd
import os
import sys

In [2]:
# Brightway imports
import bw2analyzer as ba
import bw2calc as bc
import bw2data as bd
import bw2io as bi
import brightway2 as bw

In [3]:
BW_PROJECT = 'metallican_new' # insert your project name here
bd.projects.set_current(BW_PROJECT)
bd.databases

Databases dictionary with 2 object(s):
	biosphere3
	ecoinvent-3.10-cutoff

# Regioinvent

In [4]:
# change the path here to wherever you stored the Regioinvent Python package
sys.path.append(r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\Modules\Regioinvent\src")
import regioinvent

In [5]:
regio = regioinvent.Regioinvent(bw_project_name='metallican_new', ecoinvent_database_name='ecoinvent-3.10-cutoff', ecoinvent_version='3.10')

In [6]:
regio.spatialize_my_ecoinvent()

2026-01-20 20:18:15,591 - Regioinvent - INFO - Creating spatialized biosphere flows...
Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:05


Title: Writing activities to SQLite3 database:
  Started: 01/20/2026 20:18:17
  Finished: 01/20/2026 20:18:22
  Total time elapsed: 00:00:05
  CPU %: 98.30
  Memory %: 1.02


2026-01-20 20:26:25,512 - Regioinvent - INFO - Extracting ecoinvent to wurst...


Getting activity data


100%|██████████| 23523/23523 [00:00<00:00, 40197.31it/s]


Adding exchange data to activities


100%|██████████| 743409/743409 [00:59<00:00, 12424.87it/s]


Filling out exchange data


100%|██████████| 23523/23523 [00:05<00:00, 4556.92it/s]
2026-01-20 20:27:41,178 - Regioinvent - INFO - Spatializing ecoinvent...
Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:01:17


Title: Writing activities to SQLite3 database:
  Started: 01/20/2026 20:27:46
  Finished: 01/20/2026 20:29:03
  Total time elapsed: 00:01:17
  CPU %: 97.60
  Memory %: 19.87


In [7]:
regio.import_fully_regionalized_impact_method(lcia_method='all')

2026-01-20 20:32:16,404 - Regioinvent - INFO - Importing all available fully regionalized lcia methods for ecoinvent3.10.


In [8]:
regio.regionalize_ecoinvent_with_trade(trade_database_path=r'C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\Modules\Regioinvent\trade_data.db',
                                       regioinvent_database_name='Regioinvent',
                                       cutoff=0.99)

2026-01-20 20:33:09,896 - Regioinvent - INFO - Extracting and formatting trade data...
2026-01-20 20:33:43,688 - Regioinvent - INFO - Regionalizing main inputs of internationally-traded products of ecoinvent...
100%|██████████| 1982/1982 [15:43<00:00,  2.10it/s] 
2026-01-20 20:49:26,938 - Regioinvent - INFO - Regionalizing main inputs of non-internationally traded processes of ecoinvent...
100%|██████████| 67/67 [17:34<00:00, 15.74s/it]
2026-01-20 21:07:01,251 - Regioinvent - INFO - Creating consumption markets for internationally-traded products...
100%|██████████| 1982/1982 [1:11:51<00:00,  2.18s/it]
2026-01-20 22:18:56,038 - Regioinvent - INFO - Link regioinvent processes to each other...
2026-01-20 22:19:23,313 - Regioinvent - INFO - Aggregate duplicates together...
2026-01-20 22:19:38,040 - Regioinvent - INFO - Regionalizing the elementary flows of the regioinvent database...
2026-01-20 22:19:49,621 - Regioinvent - INFO - Write regioinvent database to brightway...
Writing activiti

Title: Writing activities to SQLite3 database:
  Started: 01/20/2026 22:19:54
  Finished: 01/20/2026 22:27:13
  Total time elapsed: 00:07:18
  CPU %: 99.30
  Memory %: 32.21


2026-01-20 22:38:41,884 - Regioinvent - INFO - Connecting ecoinvent to regioinvent processes...


# Custom LCIs

In [6]:
lci_brgm_path = r'data/LCI/Lai_2025'
lci_premise_path = r'data/LCI/from_premise'

## From BRGM

In [7]:
lci_brgm_files = [
    # "Ag_LCIs_BW_Final_v2.xlsx",
    # "Al_LCIs_BW_Final_v1.xlsx",
    # "Au_LCIs_BW_Final_v1.xlsx",
    # "Cd_LCIs_BW_Final_v1.xlsx",
    # "Cu_LCIs_BW_Final_v2.xlsx",
    # "Fe_LCIs_BW_Final_v1.xlsx",
    # "Mo_LCIs_BW_Final_v1.xlsx",
    # "Ni_LCIs_BW_Final_v1.xlsx",
    # "Zn_LCIs_BW_Final_v2.xlsx",
    # "Te_LCIs_BW_Final_v2.xlsx",

    "Graphite_LCIs_BW_Final_v1.xlsx",
    "RE_LCIs_BW_Final_v3.xlsx",
    "Li_LCIs_BW_Final_v1.xlsx",
    "Co_LCIs_BW_Final_v1.xlsx"
]

In [8]:
for filename in lci_brgm_files:
    filepath = os.path.join(lci_brgm_path, filename)
    print(f"Importing: {filename}")

    importer = bw.ExcelImporter(filepath)
    importer.apply_strategies()

    importer.match_database("ecoinvent-3.10-cutoff", fields=("name", "reference product", "unit", "location"))
    importer.match_database("biosphere3", fields=("name", "unit", "categories"))

    unlinked = list(importer.unlinked)
    print(f"  → Unlinked exchanges: {len(unlinked)}")

    if not unlinked:
        importer.write_database()
        print(f"  → Database '{importer}' written.")
    else:
        print(f"  → Skipping '{filename}' due to unlinked exchanges.")

Importing: Graphite_LCIs_BW_Final_v1.xlsx
Extracted 3 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 6.03 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
  → Unlinked exchanges: 0


Writing activities to SQLite3 database:
0% [#########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 01/15/2026 17:23:18
  Finished: 01/15/2026 17:23:18
  Total time elapsed: 00:00:00
  CPU %: 97.70
  Memory %: 1.76
Created database: Graphite
  → Database '<bw2io.importers.excel.ExcelImporter object at 0x000001BD9C9F0E20>' written.
Importing: RE_LCIs_BW_Final_v3.xlsx
Extracted 6 worksheets in 0.42 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_ze

Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 01/15/2026 17:24:17
  Finished: 01/15/2026 17:24:17
  Total time elapsed: 00:00:00
  CPU %: 90.80
  Memory %: 1.93
Created database: Rare Earths
  → Database '<bw2io.importers.excel.ExcelImporter object at 0x000001BD9CB61C60>' written.
Importing: Li_LCIs_BW_Final_v1.xlsx
Extracted 10 worksheets in 0.09 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_kee

Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 01/15/2026 17:27:16
  Finished: 01/15/2026 17:27:16
  Total time elapsed: 00:00:00
  CPU %: 100.20
  Memory %: 1.87
Created database: Lithium
  → Database '<bw2io.importers.excel.ExcelImporter object at 0x000001BD9CB63310>' written.
Importing: Co_LCIs_BW_Final_v1.xlsx
Extracted 8 worksheets in 0.09 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_ze

Writing activities to SQLite3 database:
0% [############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 01/15/2026 17:27:25
  Finished: 01/15/2026 17:27:25
  Total time elapsed: 00:00:00
  CPU %: 99.70
  Memory %: 1.91
Created database: Cobalt
  → Database '<bw2io.importers.excel.ExcelImporter object at 0x000001BD84E53820>' written.


## From Premise

In [ ]:
lci_premise_files = [
    "lci-sulfidic-tailings.xlsx",
    "lci-steel.xlsx"
]

In [ ]:
for filename in lci_premise_files:
    filepath = os.path.join(lci_premise_path, filename)
    print(f"Importing: {filename}")

    importer = bw.ExcelImporter(filepath)
    importer.apply_strategies()

    importer.match_database("ecoinvent-3.10-cutoff", fields=("name", "reference product", "unit", "location"))
    importer.match_database("biosphere3", fields=("name", "unit", "categories"))

    unlinked = list(importer.unlinked)
    print(f"  → Unlinked exchanges: {len(unlinked)}")

    if not unlinked:
        importer.write_database()
        print(f"  → Database '{importer}' written.")
    else:
        print(f"  → Skipping '{filename}' due to unlinked exchanges.")

In [ ]:
# LCI from ??
tailings = 'data/LCI/from_premise/lci-sulfidic-tailings.xlsx'
tailings = bi.ExcelImporter(tailings)
# Apply the necessary strategies
tailings.apply_strategies()

In [ ]:
# Let's try to link them with EI
tailings.match_database("ecoinvent-3.10-cutoff", fields=('name', 'reference product', 'unit', 'location'))
tailings.match_database("biosphere3", fields=('name', 'unit', 'categories'))
tailings.statistics()

In [ ]:
[u for u in tailings.unlinked if u["type"] == "technosphere"]

In [ ]:
[u for u in tailings.unlinked if u["type"] == "biosphere"]

In [ ]:
migration_tailings = {
    "fields": ["name", "reference product", "location", "categories"],
    "data": [
        (
            ("cement production, blast furnace slag 70-100%",
             "cement, blast furnace slag 70-100%",
             "US"),
            {"name": "cement production, type S",
             "reference product": "cement, type S"}
        ),

        (
            ("treatment of wastewater, average, capacity 1E9l/year",
             "wastewater, average",
             "Europe without Switzerland"),
            {"name": "treatment of wastewater, average, wastewater treatment"}
        ),

        (
            ("market for sodium hydroxide, without water, in 50% solution state",
             "sodium hydroxide, without water, in 50% solution state",
             "GLO"),
            {"location": "RoW"}
        ),
    ],
}

In [ ]:
bi.Migration(name="tailings").write(data=migration_tailings, description="ei 3.8 to 3.10")

In [ ]:
tailings.data = bi.strategies.migrate_exchanges(
    db=tailings.data,
    migration="tailings"
)

In [ ]:
tailings.match_database("ecoinvent-3.10-cutoff", fields=('name', 'reference product', 'unit', 'location'))
tailings.match_database("biosphere3", fields=('name', 'unit', 'categories'))
tailings.statistics()

In [ ]:
if len(list(tailings.unlinked)) == 0:
    tailings.write_database()

## From Istrate et al (2024)

In [8]:
from core.lca_calculation_functions import create_pedigree_matrix

In [5]:
# Import LIB raw materials LCIs
lci_lib_rms = bw.ExcelImporter(r'data/LCI/Istrate_2024/lci_LIB_raw_materials.xlsx')
lci_lib_rms.apply_strategies()
lci_lib_rms.match_database("ecoinvent-3.10-cutoff", fields=('name', 'reference product', 'unit', 'location'))
lci_lib_rms.match_database("biosphere3", fields=('name', 'unit', 'categories'))

Extracted 7 worksheets in 0.56 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 5.63 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [6]:
lci_lib_rms.statistics()
[u for u in lci_lib_rms.unlinked if u["type"] == "biosphere"]
[u for u in lci_lib_rms.unlinked if u["type"] == "technosphere"]

31 datasets
498 exchanges
0 unlinked exchanges
  


[]

In [9]:
# Add uncertainty data for pedigree matrix
for ds in lci_lib_rms.data:
    for exc in ds["exchanges"]:
        if "pedigree" in exc:
            # Pedigree are stored as strings
            pedigree_str = exc["pedigree"].strip("()")
            pedigre_scores = tuple([int(x) for x in pedigree_str.split(", ")])
            exc_amount = exc["amount"]

            uncertainty_dict = create_pedigree_matrix(pedigre_scores, exc_amount)
            exc.update(uncertainty_dict)

In [10]:
if len(list(lci_lib_rms.unlinked)) == 0:
    lci_lib_rms.write_database()

Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 10/31/2025 14:22:53
  Finished: 10/31/2025 14:22:53
  Total time elapsed: 00:00:00
  CPU %: 57.10
  Memory %: 1.34
Created database: LIB raw materials


# Export EI LCIs for selected activities

In [9]:
from core.lca_calculation_functions import get_inventory_dataset, export_activity_exchanges

In [15]:
EI_DB = 'ecoinvent-3.10-cutoff'
CU_DB = 'Copper'
NI_DB = 'Nickel'
FE_DB = 'Iron'

In [11]:
INVENTORIES_EI = {
    "Cu concentrate from Au": ("gold-silver mine operation and beneficiation", "copper concentrate, sulfide ore", 'CA-QC'),
    "Cu concentrate": ("copper mine operation and beneficiation, sulfide ore", "copper concentrate, sulfide ore", 'CA'),
    "Fe concentrate": ("iron ore mine operation and beneficiation", "iron ore concentrate", "CA-QC"),
    "Au-Ag ingot": ("gold-silver mine operation and beneficiation", "gold-silver, ingot", "CA-QC"),
    #"Au": ("gold-silver mine operation with refinery", "gold", "CA-QC"),
    "Ni concentrate": ("nickel mine operation and benefication to nickel concentrate, 16% Ni", "nickel concentrate, 16% Ni", "CA-QC"),
}

In [16]:
INVENTORIES_CU = {
    "Cu concentrate from SE (Sanjuan-Delmás)": ("[M+C] Copper (Cu) ore mining and concentration (Sweden)", "Cu concentrate", 'SE'),
    "Cu concentrate from AU (Norgate)": ("[M+C] Copper (Cu) ore mining and concentration (Australia)", "Cu concentrate", 'AU'),
}

In [17]:
INVENTORIES_NI = {
    "Ni concentrate from AU (Wei)": ("[M+C] Nickel (Ni) ore mining and concentration (for Ni Class 1)", "Ni concentrate", 'AU'),
}

In [12]:
INVENTORIES_EI_ds = get_inventory_dataset(INVENTORIES_EI, database_names=[EI_DB])

In [18]:
INVENTORIES_CU_ds = get_inventory_dataset(INVENTORIES_CU, database_names=[CU_DB])

In [19]:
INVENTORIES_NI_ds = get_inventory_dataset(INVENTORIES_NI, database_names=[NI_DB])

In [13]:
INVENTORIES_EI_ds

{'Cu concentrate from Au': 'gold-silver mine operation and beneficiation' (kilogram, CA-QC, None),
 'Cu concentrate': 'copper mine operation and beneficiation, sulfide ore' (kilogram, CA, None),
 'Fe concentrate': 'iron ore mine operation and beneficiation' (kilogram, CA-QC, None),
 'Au-Ag ingot': 'gold-silver mine operation and beneficiation' (kilogram, CA-QC, None),
 'Ni concentrate': 'nickel mine operation and benefication to nickel concentrate, 16% Ni' (kilogram, CA-QC, None)}

In [14]:
# 2️⃣ Export their technosphere & biosphere flows
export_activity_exchanges(INVENTORIES_EI_ds)

✅ Export complete: CSVs saved in 'exports/'


In [20]:
export_activity_exchanges(INVENTORIES_CU_ds)

✅ Export complete: CSVs saved in 'exports/'


In [22]:
export_activity_exchanges(INVENTORIES_NI_ds)

✅ Export complete: CSVs saved in 'exports/'
